# 🎬 الحي 404 — الأصوات المصرية من الموبايل (بدون اشتراكات)

الدفتر ده بيشتغل من **Chrome على الموبايل** عن طريق Google Colab المجاني. الموبايل بيكون شاشة تحكم فقط، والمعالجة بتتم على Google GPU لما تكون متاحة.

**الخطوات:**
1. افتح الدفتر من الرابط في ChatGPT، وسجّل بحساب Google. لو الأزرار صغيرة، اختار **عرض موقع الكمبيوتر / Desktop site** من إعدادات Chrome.
2. من **Runtime → Change runtime type** اختر **T4 GPU** أو GPU المتاح؛ إن لم يوجد، مش هنقدر نضمن التجربة المجانية.
3. شغّل **خلية التجهيز** ثم **خلية سماع أصوات ليلى ووداد وصابر** بالترتيب. ممكن تشغّل كل خلية بزر التشغيل ▶.
4. فقط لو الأصوات مصرية طبيعية ومناسبة للشخصيات، فعّل مربع **اعتماد الأصوات** بآخر خلية وشغّلها لتنزيل ملف الأصوات التسعة ZIP.

**مهم:** النتيجة 9 ملفات صوت بالحوار الأصلي، **مش فيديو 3D أو ليب-سينك**. ده أول جزء من خط إنتاج الحلقة. مفيش محتوى مدفوع، لكن موارد Google Colab المجانية مش مضمونة وممكن تنتهي أثناء التشغيل.

**للمشاهدة قبل التحميل:** افتح رابط معاينات النموذج الأصلي https://mohammedaly22.github.io/VoiceTuT-TTS/ .

**حقوق الاستخدام:** الأصوات المدمجة (Yasmin / Hanan / Shaker) مخصصة للتجربة، ومطوّر VoiceTut لا يمنح حقوق استخدام تجاري لهذه الأصوات. لو المسلسل هيتنشر بشكل مربح، لازم نستخدم تسجيلات صوتية عندك حق استخدامها بدل الأصوات دي. [توضيح المطوّر](https://github.com/MohammedAly22/VoiceTuT-TTS/issues/2).


In [ ]:
# 1) تجهيز تشغيل مجاني على GPU — مع رفض التشغيل الثقيل إذا لم تتوفر GPU.
import os, subprocess, sys
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU مش متاحة الآن. افتح Runtime → Change runtime type واختار GPU لو موجود، ثم شغّل الخلية مرة تانية.")
print("GPU:", torch.cuda.get_device_name(0))
if not Path("/content/404/.git").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/omarnet0109-jpg/404.git","/content/404"],check=True)
os.chdir("/content/404")
subprocess.run([sys.executable,"-m","pip","install","-q","git+https://github.com/k2-fsa/OmniVoice.git"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","voicetut-tts"],check=True)
print("✅ جاهز لاختبار أصوات الشخصيات. شغّل الخلية التالية.")


In [ ]:
# 2) اسمع صوت كل شخصية بالمصري. شغّل الخلية بعد التجهيز.
import subprocess, sys
from IPython.display import Audio, display
subprocess.run([sys.executable,"production/free_tts.py","--preview"],check=True)
for name_ar,person in [("ليلى","layla"),("الحاجة وداد","widad"),("عم صابر","saber")]:
    print("صوت:",name_ar)
    display(Audio(filename=f"renders/episode-01/voice_previews/{person}.wav"))


## 3) اعتماد الأصوات بعد الاستماع

لو الثلاثة أصوات مناسبة للهجة المصرية ولأعمار الشخصيات، علّم على خيار **اعتماد الأصوات** وشغّل الخلية اللي تحت. هتنتج الـ9 جمل كما كتبها المؤلف، وتجهز ZIP للتحميل على الموبايل.

**لو صوت غير مناسب:** متعتمدوش؛ ارجع للمحادثة وقولّي اسم الشخصية اللي صوتها محتاج يتغير. الأسماء الحالية مبدئية، ومش هعمل موافقة تلقائية نيابة عنك.

**المخرجات دي للتجربة والمراجعة الصوتية فقط، وليست أصواتًا مرخّصة تلقائيًا للنشر التجاري.**


In [ ]:
#@title 3) توليد جمل الحلقة — بعد الموافقة على الأصوات
اعتماد_الأصوات = False #@param {type:"boolean"}
if not اعتماد_الأصوات:
    print("اسمع معاينات ليلى ووداد وصابر الأول، وبعدين فعّل خيار اعتماد الأصوات.")
else:
    import json
    from pathlib import Path
    cfg=Path("config/free_voices.json")
    data=json.loads(cfg.read_text(encoding="utf-8"))
    data["voices_previewed_and_approved"]=True
    cfg.write_text(json.dumps(data,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    subprocess.run([sys.executable,"production/free_tts.py","--episode"],check=True)
    import zipfile
    zpath=Path("/content/404_episode_1_egyptian_voices.zip")
    with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(Path("renders/episode-01/voices").glob("*.wav")):
            z.write(f,arcname=f.name)
    from google.colab import files
    files.download(str(zpath))
